In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('results_metrics.csv')

DISPLAY = {'ACMA':'ACMA', 'I2MC':'I2MC', 'NH':'NH', 'REMODNAV':"REMoDNaV",
           'UnEye':"U'n'Eye", 'Engbert':'Engbert',
           'IVT':'IVT', 'IVVT':'IVVT', 'IDT':'IDT', 'IDVT':'IDVT'}

DIRECTION = {'F1':'max', 'Kappa':'max', 'Onset_ms':'min', 'Offset_ms':'min'}

def collect(method, metric, target):
    return df[(df['metric']==metric) & (df['target_class']==target)][method].values

methods_top6 = ['ACMA', 'I2MC', 'NH', 'REMODNAV', 'UnEye', 'Engbert']

def make_table_aggregate(methods):
    rows = []
    for m in methods:
        f1   = collect(m, 'F1', 'mean')
        k    = collect(m, 'Kappa', 'mean')
        on   = collect(m, 'Onset_ms', 'sac')
        off  = collect(m, 'Offset_ms', 'sac')
        rows.append({
            'Method':       DISPLAY[m],
            'F1':           f"{f1.mean():.3f} $\\pm$ {f1.std():.3f}",
            "Cohen's κ":    f"{k.mean():.3f} $\\pm$ {k.std():.3f}",
            'Onset, ms':    f"{on.mean():.2f} $\\pm$ {on.std():.2f}",
            'Offset, ms':   f"{off.mean():.2f} $\\pm$ {off.std():.2f}",
        })
    return pd.DataFrame(rows)

table1 = make_table_aggregate(methods_top6)

In [ ]:
methods_all = ['ACMA', 'I2MC', 'NH', 'REMODNAV', 'UnEye', 'Engbert',
               'IVT', 'IVVT', 'IDT', 'IDVT']

def collect_ds(method, metric, target, ds):
    sub = df[(df['metric']==metric) & (df['target_class']==target) & (df['dataset']==ds)]
    return sub[method].values

def make_table_per_dataset(methods):
    rows = []
    for ds in ['andersson', 'eyelink', 'sim']:
        for m in methods:
            f1  = collect_ds(m, 'F1', 'mean', ds)
            k   = collect_ds(m, 'Kappa', 'mean', ds)
            on  = collect_ds(m, 'Onset_ms', 'sac', ds)
            off = collect_ds(m, 'Offset_ms', 'sac', ds)
            rows.append({
                'Dataset': ds,
                'Method':  DISPLAY[m],
                'F1':      f"{f1.mean():.3f} ± {f1.std():.3f}",
                "Cohen's κ": f"{k.mean():.3f} ± {k.std():.3f}",
                'Onset, ms':  f"{on.mean():.2f} ± {on.std():.2f}",
                'Offset, ms': f"{off.mean():.2f} ± {off.std():.2f}",
            })
    return pd.DataFrame(rows)

table2 = make_table_per_dataset(methods_all)


In [ ]:
from scipy.stats import friedmanchisquare, wilcoxon

methods_compact = ['ACMA', 'I2MC', 'NH', 'REMODNAV', 'UnEye', 'Engbert']
n_baselines = len(methods_compact) - 1   
alpha_bonf = 0.05 / n_baselines          

def build_matrix(metric, target):
    sub = df[(df['metric']==metric) & (df['target_class']==target)]
    sub = sub.sort_values(['dataset','RMS_Noise'])
    return sub[methods_compact].values

print("="*70)
print(f"FRIEDMAN TEST (n=60), {len(methods_compact)} methods")
print("="*70)
for metric, target in [('F1','mean'), ('Kappa','mean'),
                       ('Onset_ms','sac'), ('Offset_ms','sac')]:
    M = build_matrix(metric, target)
    stat, p = friedmanchisquare(*[M[:,i] for i in range(M.shape[1])])
    print(f"  {metric:10s}: χ²={stat:7.2f},  p={p:.2e}")

print()
print("="*70)
print(f"WILCOXON: ACMA vs each baseline   |  Bonferroni α = 0.05/{n_baselines} = {alpha_bonf:.3f}")
print("="*70)
for metric, target in [('F1','mean'), ('Kappa','mean'),
                       ('Onset_ms','sac'), ('Offset_ms','sac')]:
    print(f"\n  --- {metric} ---")
    direction = DIRECTION[metric]
    sub = df[(df['metric']==metric) & (df['target_class']==target)]
    sub = sub.sort_values(['dataset','RMS_Noise'])
    a = sub['ACMA'].values
    for b in [m for m in methods_compact if m != 'ACMA']:
        x = sub[b].values
        try:
            stat, p = wilcoxon(a, x,
                               alternative='greater' if direction=='max' else 'less')
            sign = "✓ ACMA wins" if p < alpha_bonf else "  n.s."
            print(f"    ACMA vs {DISPLAY[b]:10s}: W={stat:8.0f}, p={p:.2e}  {sign}")
        except ValueError as e:
            print(f"    ACMA vs {DISPLAY[b]:10s}: skipped ({e})")



In [ ]:
methods_compact = ['ACMA', 'I2MC', 'NH', 'REMODNAV', 'UnEye', 'Engbert']
regimes = [('Low (RMS 1–13)', 1, 13),
           ('Mid (RMS 15–25)', 15, 25),
           ('High (RMS 27–39)', 27, 39)]

rows = []
for label, lo, hi in regimes:
    row = {'Noise regime': label}
    sub_f1 = df[(df['metric']=='F1') & (df['target_class']=='mean') &
                (df['RMS_Noise']>=lo) & (df['RMS_Noise']<=hi)]
    sub_k  = df[(df['metric']=='Kappa') & (df['target_class']=='mean') &
                (df['RMS_Noise']>=lo) & (df['RMS_Noise']<=hi)]
    for m in methods_compact:
        row[f"{DISPLAY[m]} F1"] = f"{sub_f1[m].mean():.3f}"
        row[f"{DISPLAY[m]} κ"]  = f"{sub_k[m].mean():.3f}"
    rows.append(row)

regime_table = pd.DataFrame(rows)

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('results_metrics.csv')

DISPLAY = {'ACMA':'ACMA', 'I2MC':'I2MC', 'NH':'NH', 'REMODNAV':"REMoDNaV",
           'UnEye':"U'n'Eye", 'Engbert':'Engbert'}

methods_top6 = ['ACMA', 'I2MC', 'NH', 'REMODNAV', 'UnEye', 'Engbert']

def collect_ds_class(method, metric, target, ds):
    sub = df[(df['metric']==metric) & (df['target_class']==target) & (df['dataset']==ds)]
    return sub[method].values

def make_table_per_class(methods):
    rows = []
    for ds in ['andersson', 'eyelink', 'sim']:
        for m in methods:
            sac_f1 = collect_ds_class(m, 'F1', 'sac', ds)
            sac_k  = collect_ds_class(m, 'Kappa', 'sac', ds)
            fix_f1 = collect_ds_class(m, 'F1', 'fix', ds)
            fix_k  = collect_ds_class(m, 'Kappa', 'fix', ds)
            rows.append({
                'Dataset': ds,
                'Method':  DISPLAY[m],
                'Sac F1':  f"{sac_f1.mean():.3f} ± {sac_f1.std():.3f}",
                'Sac κ':   f"{sac_k.mean():.3f} ± {sac_k.std():.3f}",
                'Fix F1':  f"{fix_f1.mean():.3f} ± {fix_f1.std():.3f}",
                'Fix κ':   f"{fix_k.mean():.3f} ± {fix_k.std():.3f}",
            })
    return pd.DataFrame(rows)

table3 = make_table_per_class(methods_top6)